<a href="https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session5/solutions/khcc_langgraph_multiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# KHCC Hospital Administration — LangGraph Multi-Agent System

A clinical workflow with specialist agents, tools, shared state, and physician human-in-the-loop approval. This notebook implements the architecture from the diagram:

```
START → Intake → Triage Supervisor ──┬─► Lab Analyst ────┐
                                     ├─► Radiology ─────┤
                                     ├─► Pharmacy ──────┤
                                     ├─► Medical Records┤→ Synthesizer
                                     ├─► Scheduling ────┤
                                     └─► Billing ───────┘

                 Synthesizer → [HITL interrupt: Attending Physician]
                     ▲                │
                     │ rejected       │ approved
                     └────────────────┤
                                      ▼
                              Action Executor → END
```

**Key design choices from the diagram:**
- Shared `TypedDict` state with `add_messages` reducer
- `create_agent()` (LangChain 1.0+) for each specialist with its own tools
- `Send` API for parallel fan-out across specialist + supporting lanes
- `interrupt()` + `Command(resume=...)` for physician approval
- `SqliteSaver` checkpointer (thread_id = patient_id) + `InMemoryStore` for long-term facts (allergies, language, care team)
- Rejection loop back to the Synthesizer

> ⚠️ All tools are mocked — no real PHI, no real EHR writes. Replace with your KHCC VISTA / Azure SQL tools for production.

## 1. Install dependencies

In [ ]:
%pip install -q \
    "langchain>=1.0,<2.0" \
    "langchain-core>=1.0,<2.0" \
    "langchain-openai" \
    "langgraph>=1.0,<2.0" \
    "langgraph-checkpoint-sqlite"

## 2. Configure the LLM

Set your OpenAI key, or swap in Azure OpenAI (`gpt-4.1-mini` on `openai-aidi.openai.azure.com`) for the AIDI stack.

In [ ]:
import os, getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

from langchain.chat_models import init_chat_model

# For OpenAI:
MODEL = init_chat_model("openai:gpt-4o-mini", temperature=0)

# For Azure OpenAI (AIDI), uncomment and fill in:
# from langchain_openai import AzureChatOpenAI
# MODEL = AzureChatOpenAI(
#     azure_deployment="gpt-4.1-mini",
#     azure_endpoint="https://openai-aidi.openai.azure.com/",
#     api_version="2024-08-01-preview",
#     temperature=0,
# )

## 3. Shared state (TypedDict)

All nodes read the full state but return only the keys they changed. `messages` uses `add_messages` so it's appended, never replaced. List results from parallel workers also use `operator.add` reducers so nothing overwrites.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
import operator
from langgraph.graph.message import add_messages

Urgency = Literal["routine", "urgent", "emergency"]
Approval = Literal["pending", "approved", "rejected"]

class KHCCState(TypedDict, total=False):
    # Inputs
    patient_id: str
    chief_complaint: str

    # Triage
    urgency: Urgency

    # Specialist outputs (single-writer fields — no reducer needed)
    lab_summary: str
    imaging_summary: str
    medication_list: Annotated[list[str], operator.add]
    allergies: Annotated[list[str], operator.add]
    history_notes: str
    scheduling_note: str
    billing_note: str

    # Downstream
    recommendation: str
    approval_status: Approval
    physician_feedback: str
    final_actions: Annotated[list[str], operator.add]

    # Conversation
    messages: Annotated[list, add_messages]

## 4. Mock clinical tools

These stand in for real VISTA / EHR calls. Each is a small deterministic function so the notebook runs end-to-end without any backend. Swap in real tools as you productionize.

In [ ]:
from langchain_core.tools import tool

# ------------------ Intake ------------------
@tool
def get_patient_record(patient_id: str) -> dict:
    """Fetch the basic demographic + chief-complaint record for a patient."""
    return {
        "patient_id": patient_id,
        "name": "<REDACTED>",
        "age": 58,
        "sex": "F",
        "chief_complaint": "Fever and neutropenia post chemotherapy",
        "vitals": {"T": 38.9, "HR": 112, "BP": "98/62", "SpO2": 94},
    }

# ------------------ Lab ------------------
@tool
def fetch_labs(patient_id: str) -> dict:
    """Fetch most recent CBC and chemistry for a patient."""
    return {
        "ANC": 0.3,  # neutropenic
        "WBC": 1.1,
        "Hgb": 8.9,
        "Plt": 78,
        "Creatinine": 1.4,
        "CRP": 180,
    }

@tool
def check_reference_ranges(labs: dict) -> dict:
    """Flag lab values outside reference ranges."""
    flags = []
    if labs.get("ANC", 1.5) < 0.5: flags.append("severe neutropenia")
    if labs.get("Plt", 200) < 100: flags.append("thrombocytopenia")
    if labs.get("Creatinine", 0.8) > 1.2: flags.append("elevated creatinine")
    if labs.get("CRP", 0) > 50: flags.append("elevated CRP")
    return {"flags": flags}

# ------------------ Radiology ------------------
@tool
def fetch_imaging_report(patient_id: str) -> str:
    """Fetch the most recent imaging report for a patient."""
    return (
        "Chest CT (today): No consolidation. Small bilateral pleural effusions. "
        "No pulmonary embolism. Port-a-cath tip in SVC without thrombus."
    )

@tool
def summarize_findings(report: str) -> str:
    """Summarize a raw imaging report into 1-2 clinical lines."""
    return f"Summary: {report[:180]}..."

# ------------------ Pharmacy ------------------
@tool
def check_drug_interaction(current_meds: list[str], proposed: str) -> dict:
    """Check proposed drug against current medications for interactions."""
    risky = {("warfarin", "ciprofloxacin"), ("methotrexate", "trimethoprim")}
    interactions = [
        f"{m} × {proposed}"
        for m in current_meds
        if (m.lower(), proposed.lower()) in risky
    ]
    return {"interactions": interactions}

@tool
def get_formulary_dose(drug: str, weight_kg: float = 70) -> str:
    """Return the KHCC formulary dose for a drug."""
    doses = {
        "piperacillin-tazobactam": "4.5 g IV q6h",
        "vancomycin": f"{round(15*weight_kg)} mg IV q12h (renal adj.)",
        "meropenem": "1 g IV q8h",
    }
    return doses.get(drug.lower(), f"No formulary entry for {drug}")

# ------------------ Medical Records ------------------
@tool
def get_allergies(patient_id: str) -> list[str]:
    """Return the patient's known drug allergies."""
    return ["penicillin (rash)"]

@tool
def get_history(patient_id: str) -> str:
    """Return the patient's relevant medical history."""
    return "Stage III ovarian cancer, on carboplatin-paclitaxel. Last chemo 9 days ago."

# ------------------ Scheduling ------------------
@tool
def book_appointment(patient_id: str, service: str, when: str) -> str:
    """Book a follow-up appointment."""
    return f"Appointment booked for {patient_id}: {service} at {when}"

@tool
def find_OR_slot(urgency: str) -> str:
    """Find an available OR slot."""
    return "OR-3 at 14:00 (urgent case)"

# ------------------ Billing ------------------
@tool
def verify_coverage(patient_id: str) -> dict:
    """Verify insurance coverage."""
    return {"insurer": "Jordan Social Security", "status": "active", "copay": 0}

@tool
def code_ICD10(diagnosis: str) -> str:
    """Return an ICD-10 code for a diagnosis."""
    table = {
        "febrile neutropenia": "D70.9",
        "sepsis": "A41.9",
        "ovarian cancer": "C56.9",
    }
    for k, v in table.items():
        if k in diagnosis.lower():
            return v
    return "R69  (unspecified)"

# ------------------ Action Executor ------------------
@tool
def write_to_EHR(patient_id: str, note: str) -> str:
    """Write a signed note to the EHR."""
    return f"EHR note saved for {patient_id} ({len(note)} chars)"

@tool
def send_prescription(patient_id: str, drug: str, dose: str) -> str:
    """Send an e-prescription."""
    return f"Rx sent: {drug} {dose} for {patient_id}"

@tool
def notify_nurse(patient_id: str, message: str) -> str:
    """Page the primary nurse."""
    return f"Nurse paged for {patient_id}: {message}"

## 5. Specialist agents via `create_agent()`

Each specialist is a tiny ReAct-style agent built with LangChain 1.0's `create_agent()`. Each gets its own toolset and system prompt. The supervisor and synthesizer are plain LLM nodes (no tools).

In [ ]:
from langchain.agents import create_agent

intake_agent = create_agent(
    model=MODEL,
    tools=[get_patient_record],
    system_prompt=(
        "You are the intake agent at KHCC. Given a patient_id, call get_patient_record "
        "and return a one-paragraph clinical summary including chief complaint and vitals."
    ),
)

lab_agent = create_agent(
    model=MODEL,
    tools=[fetch_labs, check_reference_ranges],
    system_prompt=(
        "You are the lab analyst. Fetch labs, flag abnormal values, and write ONE concise "
        "clinical line naming the key abnormalities. Do not speculate on treatment."
    ),
)

radiology_agent = create_agent(
    model=MODEL,
    tools=[fetch_imaging_report, summarize_findings],
    system_prompt=(
        "You are the radiology agent. Fetch the most recent imaging report and return "
        "a one-line clinical summary of positive findings."
    ),
)

pharmacy_agent = create_agent(
    model=MODEL,
    tools=[check_drug_interaction, get_formulary_dose],
    system_prompt=(
        "You are the pharmacy agent. For a febrile neutropenic patient, propose an empiric "
        "antibiotic from the KHCC formulary, check interactions with current meds, and return "
        "the drug + dose as a single line."
    ),
)

records_agent = create_agent(
    model=MODEL,
    tools=[get_allergies, get_history],
    system_prompt="You are the medical records agent. Retrieve allergies and relevant history.",
)

scheduling_agent = create_agent(
    model=MODEL,
    tools=[book_appointment, find_OR_slot],
    system_prompt=(
        "You are the scheduling agent. For an urgent/emergency patient, find an admission "
        "or OR slot. Return a one-line plan."
    ),
)

billing_agent = create_agent(
    model=MODEL,
    tools=[verify_coverage, code_ICD10],
    system_prompt=(
        "You are the billing/insurance agent. Verify coverage and assign an ICD-10 code for "
        "the working diagnosis. Return a one-line compliance note."
    ),
)

executor_agent = create_agent(
    model=MODEL,
    tools=[write_to_EHR, send_prescription, notify_nurse],
    system_prompt=(
        "You are the action executor. Given an APPROVED recommendation, call the appropriate "
        "write tools (EHR note, prescription, nurse page). Report each action you took."
    ),
)

## 6. Graph nodes

Each node wraps an agent (or raw LLM call) and returns a partial state update. The triage supervisor sets `urgency`. The fan-out function uses `Send` to dispatch all specialists in parallel.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Command, Send, interrupt
from langgraph.graph import StateGraph, START, END

def _run_agent(agent, prompt: str) -> str:
    """Invoke a create_agent agent with a single user prompt; return the final text."""
    out = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    return out["messages"][-1].content

# --- Intake ---
def intake_node(state: KHCCState) -> dict:
    summary = _run_agent(
        intake_agent,
        f"Fetch and summarize patient {state['patient_id']}.",
    )
    return {
        "chief_complaint": state.get("chief_complaint") or summary,
        "messages": [SystemMessage(content=f"[Intake] {summary}")],
    }

# --- Triage Supervisor: sets urgency and fans out ---
def triage_supervisor(state: KHCCState) -> dict:
    prompt = (
        "Classify the urgency of this presentation as exactly one of: "
        "'routine', 'urgent', 'emergency'.\n\n"
        f"Chief complaint / intake: {state.get('chief_complaint','')}\n\n"
        "Reply with just the single word."
    )
    resp = MODEL.invoke(prompt).content.strip().lower()
    urgency: Urgency = "routine"
    for u in ("emergency", "urgent", "routine"):
        if u in resp:
            urgency = u  # type: ignore[assignment]
            break
    return {
        "urgency": urgency,
        "messages": [SystemMessage(content=f"[Triage] Urgency = {urgency}")],
    }

# --- Specialist lane nodes ---
def lab_node(state: KHCCState) -> dict:
    s = _run_agent(lab_agent, f"Analyze labs for patient {state['patient_id']}.")
    return {"lab_summary": s, "messages": [SystemMessage(content=f"[Lab] {s}")]}

def radiology_node(state: KHCCState) -> dict:
    s = _run_agent(radiology_agent, f"Review imaging for patient {state['patient_id']}.")
    return {"imaging_summary": s, "messages": [SystemMessage(content=f"[Rad] {s}")]}

def pharmacy_node(state: KHCCState) -> dict:
    s = _run_agent(
        pharmacy_agent,
        "Patient is febrile neutropenic with no known current antibiotics. "
        "Propose empiric coverage and dose.",
    )
    return {"medication_list": [s], "messages": [SystemMessage(content=f"[Pharm] {s}")]}

# --- Supporting lane nodes ---
def records_node(state: KHCCState) -> dict:
    s = _run_agent(records_agent, f"Pull allergies and history for patient {state['patient_id']}.")
    # Cheap parse — just stuff it in history_notes; real version would structure this
    return {
        "history_notes": s,
        "allergies": [],  # real version: parse from tool output
        "messages": [SystemMessage(content=f"[Records] {s}")],
    }

def scheduling_node(state: KHCCState) -> dict:
    if state.get("urgency") == "routine":
        s = "No urgent scheduling needed — outpatient follow-up in clinic."
    else:
        s = _run_agent(
            scheduling_agent,
            f"Find an admission slot for urgent patient {state['patient_id']}.",
        )
    return {"scheduling_note": s, "messages": [SystemMessage(content=f"[Sched] {s}")]}

def billing_node(state: KHCCState) -> dict:
    s = _run_agent(
        billing_agent,
        f"Verify coverage and code working diagnosis for patient {state['patient_id']} "
        "(working dx: febrile neutropenia in ovarian cancer).",
    )
    return {"billing_note": s, "messages": [SystemMessage(content=f"[Billing] {s}")]}

# --- Fan-out dispatcher ---
FANOUT_NODES = ["lab", "radiology", "pharmacy", "records", "scheduling", "billing"]

def dispatch(state: KHCCState):
    """Conditional-edge function: fan-out to all specialist + supporting lanes."""
    return [Send(n, state) for n in FANOUT_NODES]

# --- Synthesizer: merges everything into a single recommendation ---
def synthesizer_node(state: KHCCState) -> dict:
    feedback = state.get("physician_feedback", "")
    prior = state.get("recommendation", "")
    prompt = (
        "You are the KHCC recommendation synthesizer. Merge the inputs below into a SINGLE "
        "concise clinical recommendation (3-5 bullet points). Include: working diagnosis, "
        "next antibiotic, disposition, and ICD-10 code.\n\n"
        f"Urgency: {state.get('urgency')}\n"
        f"Lab: {state.get('lab_summary','')}\n"
        f"Imaging: {state.get('imaging_summary','')}\n"
        f"Pharmacy: {state.get('medication_list', [])}\n"
        f"History: {state.get('history_notes','')}\n"
        f"Scheduling: {state.get('scheduling_note','')}\n"
        f"Billing: {state.get('billing_note','')}\n"
    )
    if feedback:
        prompt += (
            f"\nPHYSICIAN FEEDBACK on previous draft: {feedback}\n"
            f"Previous draft was: {prior}\n"
            "Revise the recommendation to address this feedback."
        )
    rec = MODEL.invoke(prompt).content
    return {
        "recommendation": rec,
        "approval_status": "pending",
        "physician_feedback": "",  # clear after consumption
        "messages": [SystemMessage(content=f"[Synthesizer] {rec}")],
    }

# --- HITL: Attending Physician Approval ---
def physician_review_node(state: KHCCState) -> Command:
    """Pause the graph and wait for physician decision.

    Resume with:
        Command(resume={"approval_status": "approved"})
      or
        Command(resume={"approval_status": "rejected", "feedback": "Use meropenem instead"})
    """
    decision = interrupt({
        "draft_recommendation": state.get("recommendation", ""),
        "prompt": "Attending physician: approve or reject this recommendation.",
    })
    status = decision.get("approval_status", "rejected")
    feedback = decision.get("feedback", "")

    if status == "approved":
        return Command(
            update={"approval_status": "approved",
                    "messages": [SystemMessage(content="[HITL] Approved by attending.")]},
            goto="action_executor",
        )
    else:
        return Command(
            update={"approval_status": "rejected",
                    "physician_feedback": feedback,
                    "messages": [SystemMessage(content=f"[HITL] Rejected: {feedback}")]},
            goto="synthesizer",  # loop back
        )

# --- Action Executor ---
def action_executor_node(state: KHCCState) -> dict:
    prompt = (
        f"Execute the following APPROVED plan for patient {state['patient_id']}.\n\n"
        f"{state['recommendation']}\n\n"
        "Use write_to_EHR for the note, send_prescription for each antibiotic, "
        "and notify_nurse with a one-line handoff. List every action you took."
    )
    out = _run_agent(executor_agent, prompt)
    return {
        "final_actions": [out],
        "messages": [SystemMessage(content=f"[Executor] {out}")],
    }

## 7. Wire up the graph

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.memory import InMemoryStore

# Long-term cross-thread store (allergies, language, care team)
store = InMemoryStore()
store.put(("patient-demo-001", "preferences"), "language", {"pref": "Arabic primary, English ok"})
store.put(("patient-demo-001", "care_team"), "attending", {"name": "Dr. Ibdah"})

# Thread-scoped checkpointer (use :memory: for the demo; swap to a file path for persistence)
conn = sqlite3.connect(":memory:", check_same_thread=False)
checkpointer = SqliteSaver(conn)

builder = StateGraph(KHCCState)

# Register nodes
builder.add_node("intake", intake_node)
builder.add_node("triage", triage_supervisor)
builder.add_node("lab", lab_node)
builder.add_node("radiology", radiology_node)
builder.add_node("pharmacy", pharmacy_node)
builder.add_node("records", records_node)
builder.add_node("scheduling", scheduling_node)
builder.add_node("billing", billing_node)
builder.add_node("synthesizer", synthesizer_node)
builder.add_node("physician_review", physician_review_node)
builder.add_node("action_executor", action_executor_node)

# Edges
builder.add_edge(START, "intake")
builder.add_edge("intake", "triage")

# Fan-out from triage to all six lanes (conditional edge returning Send list)
builder.add_conditional_edges("triage", dispatch, FANOUT_NODES)

# All lanes converge on the synthesizer
for n in FANOUT_NODES:
    builder.add_edge(n, "synthesizer")

builder.add_edge("synthesizer", "physician_review")
# physician_review routes via Command(goto=...)
builder.add_edge("action_executor", END)

graph = builder.compile(checkpointer=checkpointer, store=store)
print("✅ Graph compiled")

## 8. Visualize the graph

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # Fallback: print Mermaid source
    print(graph.get_graph().draw_mermaid())

## 9. Run: first turn — stops at the HITL interrupt

`thread_id = patient_id` so every checkpoint is scoped to one patient's case.

In [ ]:
PATIENT_ID = "patient-demo-001"
config = {"configurable": {"thread_id": PATIENT_ID}, "recursion_limit": 50}

result = graph.invoke(
    {
        "patient_id": PATIENT_ID,
        "chief_complaint": "58F with fever 38.9 and neutropenia post chemo",
        "messages": [],
    },
    config=config,
)

print("=== URGENCY ===")
print(result.get("urgency"))
print()
print("=== DRAFT RECOMMENDATION (awaiting physician) ===")
print(result.get("recommendation"))
print()
print("=== INTERRUPT PAYLOAD ===")
print(result.get("__interrupt__"))

## 10. Resume: physician REJECTS with feedback → back to synthesizer

This demonstrates the rejection feedback loop. The synthesizer re-runs with the feedback baked in, and the graph pauses again at HITL for a second review.

In [ ]:
result = graph.invoke(
    Command(resume={
        "approval_status": "rejected",
        "feedback": "Patient has penicillin allergy — switch from piperacillin-tazobactam to meropenem. Also add vancomycin for MRSA coverage.",
    }),
    config=config,
)

print("=== REVISED RECOMMENDATION ===")
print(result.get("recommendation"))
print()
print("=== INTERRUPT PAYLOAD (2nd review) ===")
print(result.get("__interrupt__"))

## 11. Resume: physician APPROVES → Action Executor → END

In [ ]:
result = graph.invoke(
    Command(resume={"approval_status": "approved"}),
    config=config,
)

print("=== FINAL APPROVAL STATUS ===")
print(result.get("approval_status"))
print()
print("=== EXECUTED ACTIONS ===")
for a in result.get("final_actions", []):
    print("—", a)
print()
print("=== MESSAGE TRACE ===")
for m in result.get("messages", []):
    content = getattr(m, "content", str(m))
    if content:
        print("•", content[:200])

## 12. Inspect checkpoint history (time travel)

Every super-step was saved — you can replay or fork from any point.

In [ ]:
history = list(graph.get_state_history(config))
print(f"Total checkpoints: {len(history)}")
for i, snap in enumerate(history[:8]):
    print(f"[{i}] next={snap.next}  approval={snap.values.get('approval_status')}  urgency={snap.values.get('urgency')}")

## 13. Next steps for productionizing at KHCC

1. **Swap `MODEL`** to Azure OpenAI `gpt-4.1-mini` (see cell 2). Pull keys from Databricks secrets scope `sql-credentials`.
2. **Replace mock tools** with real VISTA queries — `VISTA_LAB_CHEMISTRY_2`, `vista_rad_reports`, `VISTA_PHARMACY_IV`. Use read-only connections for the specialist lanes and write-only creds for the Action Executor.
3. **Swap `SqliteSaver` → `PostgresSaver`** pointed at `aidi-db-server.database.windows.net` (or a dedicated Postgres). File-based SQLite is fine for dev but not for multi-worker Databricks jobs.
4. **Add middleware** for PHI redaction before every LLM call (`Fernet` for names, Optimus for MRNs), per-tool-call audit logging, and rate limiting per `thread_id`.
5. **Idempotency in `action_executor_node`** — before resuming after HITL, check whether the EHR note / Rx already exists (upsert-style) so a duplicate resume never double-prescribes.
6. **Trim messages** — long threads will blow the context window. Wrap `MODEL` with `trim_messages(max_tokens=4000, strategy='last', include_system=True)` in a middleware layer.
7. **Observability** — set `LANGSMITH_API_KEY` and `LANGSMITH_PROJECT=khcc-hospital-admin` to get full traces of every run. Invaluable for debugging the HITL flow.
8. **Stop conditions** — this notebook relies on the physician always responding. In production, add a timer node that auto-escalates to a backup attending after N minutes (another `interrupt` with a timeout wrapper).